In [44]:
!uv pip install datasets

Checked 1 package in 27ms


In [45]:
# load json data and get it into a format that we can use for fine-tuning and turn it into a dataset compatible with huggingface ecosystem

import json
from datasets import Dataset

with open('people_data.json', 'r') as f:
    data = json.load(f)

    # turn individual examples and turn them into formatted strings which can then be turned into a dataset that can be used for fine-tuning\
    tuning_examples = []

    for example in data:
        """
        since the data is in the shape:
        {
        "prompt": "",
        "response": {
            "name": "",
            "age": "",
            "job": "",
            "gender": ""
        }
        }
        """
        tuning_examples.append(f"<|user|>\n{example['prompt']}\n<|assistant|>\n{json.dumps(example['response'])}<|endoftext|>")


dataset = Dataset.from_dict({'text': tuning_examples})  

In [46]:
from unsloth import FastLanguageModel

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = 'unsloth/Phi-4-mini-instruct',
    max_seq_length = 2048,
    dtype = None,
    load_in_4bit = True
)

Fetching 13 files: 100%|██████████████████████████████████████████████████████████████████████████████████████| 13/13 [00:00<00:00, 50023.81it/s]


Unsloth: Loading unsloth/Phi-4-mini-instruct via mlx-lm (runtime 4-bit affine quantization)...


Fetching 13 files: 100%|██████████████████████████████████████████████████████████████████████████████████████| 13/13 [00:00<00:00, 64680.84it/s]


[INFO] Quantized model with 6.343 bits per weight.
Unsloth: Quantized text model to 4-bit affine.


In [47]:
# the rank size, r gets to determine the speed and complexity. 
# large rank size = more complex knowledge that can be learned. the smaller the rank size the faster the fine-tuning.
model = FastLanguageModel.get_peft_model(
    model,
    r = 64,
    target_modules = [
        'q_proj', 'k_proj', 'v_proj', 'o_proj', 'gate_proj', 'up_proj', 'down_proj'
    ],
    lora_alpha = 64 * 2,
    lora_dropout = 0,
    bias = 'none',
    use_gradient_checkpoint = 'unsloth'
)

Unsloth: LoRA applied — 92,274,688 trainable params (7.62% of 1,210,387,456 total)


In [48]:
# On Apple Silicon, Unsloth uses MLX — not PyTorch/TRL's SFTTrainer.
from unsloth_zoo.mlx.trainer import MLXTrainer, MLXTrainingConfig

# from trl import SFTTrainer, SFTConfig

# trainer = SFTTrainer(
#     model = model,
#     train_dataset = dataset,
#     processing_class = tokenizer,
#     args = SFTConfig(
#         max_length = 2048,
#         dataset_text_field = 'text',
#         per_device_train_batch_size = 2,
#         gradient_accumulation_steps = 4,
#         warmup_steps = 10,
#         max_steps = 60,
#         num_train_epochs = 1,
#         output_dir = 'outputs',
#         optim = 'adamw_8bit'
#     )
# )


trainer = MLXTrainer(
    model = model,
    train_dataset = dataset,
    tokenizer = tokenizer,
    args = MLXTrainingConfig(
        max_seq_length = 2048,
        dataset_text_field = 'text',
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4,
        warmup_steps = 10,
        max_steps = 60,
        num_train_epochs = 1,
        output_dir = 'outputs',
        optim = 'adamw',
    )
)

In [49]:
trainer.train()

Unsloth: MLX Metal memory guard enabled (memory_limit=14.60 GB, wired_limit=14.60 GB).
Unsloth: Using gradient checkpointing to reduce memory.
Unsloth: CCE skipping weight gradient (LM head is frozen).
Unsloth: Using CCE loss (runtime-cce) for memory-efficient training.
Unsloth: Training for 60 steps, BS=2, grad_accum=4, seq_len=2048
Unsloth: Features: CCE, GC, compile, LR=linear, opt=adamw
  Step 1/60 | Loss: 5.4826 | Grad: 53.1895 | LR: 1.82e-05 | Tok/s: 65 | Peak: 5.63 GB
  Step 2/60 | Loss: 5.0122 | Grad: 45.4062 | LR: 3.64e-05 | Tok/s: 151 | Peak: 6.09 GB
  Step 3/60 | Loss: 4.3224 | Grad: 7.7758 | LR: 5.45e-05 | Tok/s: 172 | Peak: 6.09 GB
  Step 4/60 | Loss: 4.0754 | Grad: 6.2509 | LR: 7.27e-05 | Tok/s: 185 | Peak: 6.09 GB
  Step 5/60 | Loss: 3.9204 | Grad: 5.0515 | LR: 9.09e-05 | Tok/s: 170 | Peak: 6.09 GB
  Step 6/60 | Loss: 3.4161 | Grad: 4.4638 | LR: 1.09e-04 | Tok/s: 168 | Peak: 6.09 GB
  Step 7/60 | Loss: 2.9743 | Grad: 4.3287 | LR: 1.27e-04 | Tok/s: 187 | Peak: 6.09 GB
  S

{'train_loss': 1.3666452606519064,
 'train_runtime': 178.5117553329328,
 'train_steps': 60,
 'trained_tokens': 29083,
 'train_samples_per_second': 162.91924274543624,
 'compile_enabled': True,
 'compile_support_state': 'n/a',
 'compile_reason': '',
 'compile_policy_mode': 'best_effort',
 'patch_mode': 'patched',
 'compile_trace': None,
 'compile_auto_tune_applied': [],
 'memory_limits_applied': {'memory_limit_gb': 14.602902732799999,
  'wired_limit_gb': 14.602902732799999,
  'recommended_working_set_gb': 17.179885568}}

In [56]:
from mlx_lm import generate
from mlx_lm.sample_utils import make_sampler

FastLanguageModel.for_inference(model)

messages = [
    {
        'role': 'user',
        'content': 'Mike is a 30 year old programmer. He loves hiking.'
    }
]

# CUDA/PyTorch
# inputs = tokenizer.apply_chat_template(messages, tokenize=True, add_generation_prompt=True, return_tensors='pt').to('cuda')
# outputs = model.generate(input_ids=inputs, max_new_tokens=512, use_cache=True, temperature=0.7, do_sample=True, top_p=0.9)
# response = tokenizer.batch_decode(outputs)[0]
# print(response)

prompt = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True,
)

sampler = make_sampler(
    temp=0.7,
    top_p=0.9,
)

response = generate(
    model,
    tokenizer,
    prompt=prompt,
    max_tokens=512,
    sampler=sampler,
    verbose=True,
)

print(response)

{"name": "Mike", "age": "30", "job": "programmer", "gender": "male"}<|endoftext|>
Prompt: 16 tokens, 20.471 tokens-per-sec
Generation: 27 tokens, 44.355 tokens-per-sec
Peak memory: 6.094 GB
{"name": "Mike", "age": "30", "job": "programmer", "gender": "male"}<|endoftext|>


In [64]:
# import os
# os.chdir('~/fine-tuning-with-unsloth')

# exporting to ollama
# model.save_pretrained_gguf(
#     'finetuned_model', 
#     tokenizer, 
#     quantization_method='quantized'  # maps to q4_k_m on MLX
# )

# Save the merged model
model.save_pretrained_merged('finetuned_merged', tokenizer, save_method='merged_16bit')

INFO:httpx: HTTP Request: HEAD https://huggingface.co/unsloth/Phi-4-mini-instruct/resolve/main/README.md "HTTP/1.1 307 Temporary Redirect"
INFO:httpx: HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/unsloth/Phi-4-mini-instruct/75becc471c56fc34761ec998615ca6f8535c5a61/README.md "HTTP/1.1 200 OK"


Unsloth: Merged model saved to finetuned_merged


In [65]:
import json

with open('finetuned_merged/tokenizer_config.json') as f:
    cfg = json.load(f)

cfg['tokenizer_class'] = 'PreTrainedTokenizerFast'

with open('finetuned_merged/tokenizer_config.json', 'w') as f:
    json.dump(cfg, f, indent=2)